# AgentQL + Human Pages: Human-in-the-Loop Data Extraction

This notebook demonstrates how to use the `agentql-humanpages` integration to extract structured data from web pages with an automatic human fallback.

## Setup

Install the package and set your API keys:

In [ ]:
%pip install -qU agentql-humanpages

In [ ]:
import os

os.environ["AGENTQL_API_KEY"] = "your-agentql-key"
os.environ["HUMANPAGES_API_KEY"] = "your-humanpages-key"

## Using HumanFallbackAgent

The `HumanFallbackAgent` tries AgentQL first. If extraction fails, it automatically creates a job on Human Pages.

In [ ]:
from agentql_humanpages import HumanFallbackAgent

agent = HumanFallbackAgent()

# Extract with an AgentQL query
result = agent.extract(
    url="https://www.ycombinator.com/companies",
    query="{ companies[] { name description batch } }",
)

print(f"Source: {result['source']}")
if result["source"] == "agentql":
    print(f"Data: {result['data']}")
else:
    print(f"Job ID: {result['job_id']}")
    print(f"Status: {result['status']}")

## Using Natural Language Prompts

Instead of an AgentQL query, you can use a natural language prompt:

In [ ]:
result = agent.extract(
    url="https://example.com/products",
    prompt="Extract all product names, prices, and ratings",
)
print(result)

## Using the HumanPagesClient Directly

For more control, use the `HumanPagesClient` to interact with the Human Pages API directly:

In [ ]:
from agentql_humanpages import HumanPagesClient

client = HumanPagesClient()

# Search for available humans
humans = client.search_humans(skill="web task")
print(f"Found {len(humans)} available humans")

if humans:
    # Create a job
    job = client.create_job(
        human_id=humans[0]["id"],
        title="Extract product catalog",
        description="Visit example.com/products and extract all product names and prices as JSON.",
        price_usdc=5.0,
        deadline_hours=24,
    )
    print(f"Created job: {job['id']}")